In [1]:
from torchvision.datasets import MNIST
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import numpy as np
import math


from helpers import *
from loss_and_eval import *




In [52]:
class Layer:
    def __init__(self, input_neurons, output_neurons, softmax = False):
        self.j = output_neurons
        self.k = input_neurons

        self.W = np.random.normal(scale = 1/math.sqrt(self.j), size = (self.j, self.k)).tolist()
        self.b = np.random.normal(size = (self.j, 1)).tolist()

        self.softmax = softmax

        self.clear_grad()

    def __call__(self, prev_activations):
        return self.forward(prev_activations)

    def forward(self, prev_activations):
        z = matrix_add(matrix_mult(self.W, prev_activations), self.b)
        if self.softmax:
            return vector_softmax(z), z

        return vector_sigmoid(z), z

    def clear_grad(self):
        self.W_grad = [[0.0 for _ in range(self.k)] for _ in range(self.j)]
        self.b_grad = [[0.0] for _ in range(self.j)]



class Convolution:
    def __init__(self, filters, channels, kernel_size, stride = 1):
        self.filters = filters
        self.channels = channels
        self.kernel_size = kernel_size
        self.stride = stride
 
        self.W = np.random.normal(
            size = (self.filters, self.channels, self.kernel_size, self.kernel_size)
        )
        self.b = np.random.normal(
            size = self.filters
        )

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        # x: (channels, length, width)
        input_size = x.shape[1]
        
        f_map_size = (input_size - self.kernel_size) // self.stride + 1
        f_maps = np.zeros(shape=(self.filters, f_map_size, f_map_size))

        # hard-coded
        for f in range(self.filters):
            f_map = np.zeros(shape=(f_map_size, f_map_size))

            for i in range(f_map_size):
                for j in range(f_map_size):
                    sum = self.b[f]

                    for l in range(self.kernel_size):
                        for m in range(self.kernel_size):
                            for c in range(self.channels):
                                input_channel = x[c]
                                W = self.W[f, c, :, :] # (kernel_size, kernel_size)

                                row = self.stride * i + l
                                col = self.stride * j + l
                                sum += input_channel[row][col] * W[l][m]

                    f_map[i][j] = sum

            f_maps[f] = f_map

        return f_maps


class MaxPool:
    def __init__(self, pool_size):
        self.pool_size = pool_size

    def __call__(self, f_maps):
        return self.forward(f_maps)

    def forward(self, f_maps):
        filters = f_maps.shape[0]
        f_map_size = f_maps.shape[1]

        assert f_map_size % self.pool_size == 0, f"Pool size {self.pool_size} is not valid for {f_map_size}x{f_map_size} feature maps"
        pooled_f_map_size = f_map_size // self.pool_size

        pooled_f_maps = np.zeros(shape = (filters, pooled_f_map_size, pooled_f_map_size))

        for f in range(filters):
            f_map = f_maps[f]
            pooled_f_map = np.zeros(shape = (pooled_f_map_size, pooled_f_map_size))

            for i in range(pooled_f_map_size):
                for j in range(pooled_f_map_size):
                    row = i * self.pool_size
                    col = j * self.pool_size

                    window = f_map[row:row+self.pool_size, col:col+self.pool_size]
                    pooled_f_map[i][j] = np.max(window)

            pooled_f_maps[f] = pooled_f_map

        return pooled_f_maps


